# Q3: Convolutional autoencoders: bottleneck capacity, denoising, and class information

| Part | What is done | New or reused |
|---|---|---|
| **(a)** | Three convolutional autoencoders that differ **only** in the width *d* of the bottleneck (16, 128, 1024); one is selected on validation reconstruction error | new |
| **(b)** | Corruption: additive Gaussian noise (σ = 0.2). A denoising autoencoder (DAE) with the selected architecture is trained without labels, and the **fixed CNN classifier from Q1(a)** is evaluated on corrupted images, clean-AE reconstructions and DAE reconstructions | DAE new; clean AE reused from (a); classifier **reused from Q1(a)** |
| **(c)** | Frozen-encoder **linear probe** and **k-NN probe** of class information, compared with reconstruction quality | reuses the encoders from (a) and (b) |

**Data protocol (same as Q1 and Q2).** The split is the same roll-number-seeded 45k/5k split. All selection uses validation only: the bottleneck, the
early-stopping epochs, the blur baseline and the probe hyperparameters. The test set is loaded only in Section 5,
after everything is fixed. Autoencoders work on pixel values in [0, 1] with a sigmoid output and a mean-squared-error (MSE) loss. Images
are normalised with the Q1 training-split statistics only when they are passed to the CNN classifier.

**Runtime.** Use a GPU. With the Q1 run cache available (for the classifier), the notebook trains 8 small autoencoders
(about 1–2 minutes each on a T4) plus the probes, roughly 30 minutes in total. Without the cache, it first retrains the Q1(a)
CNN learning-rate sweep (3 runs, about 10 more minutes). On Colab, point `Q1_OUT_DIR` and `OUT_DIR` to Google Drive.

In [ ]:
# ===================== 0. CONFIGURATION =====================
ROLL_NUMBER = "ENTER_YOUR_ROLL_NUMBER"   # same roll number as Q1 and Q2

DATA_ROOT  = "./data"
Q1_OUT_DIR = "./q1_outputs"   # the fixed CNN classifier is loaded from Q1(a)'s run cache
OUT_DIR    = "./q3_outputs"

# Must equal the Q1 settings: they define the cache key of the reused classifier
N_TRAIN, N_VAL = 45_000, 5_000
CFG_A = dict(epochs=30, batch_size=128, weight_decay=0.05, schedule="cosine")
LR_GRID = [3e-4, 1e-3, 3e-3]
TRAIN_EVAL_SUBSET = 5_000

# Q3(a): autoencoders (identical setup for every bottleneck)
BOTTLENECKS = [16, 128, 1024]             # 8x apart: compression 192x, 24x, 3x of the 3072 input values
CFG_AE = dict(epochs=30, batch_size=128, lr=1e-3, weight_decay=1e-4)
N_SEEDS_AE = 2

# Q3(b): corruption
NOISE_SIGMA = 0.2                          # std of additive Gaussian noise on [0, 1] pixels (~51/255)
BLUR_SIGMA_GRID = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]   # non-learned denoising baseline, chosen on validation MSE
SEVERITY_SWEEP = [0.05, 0.1, 0.2, 0.3]     # supplementary test-time severity curve

# Q3(c): probes on frozen representations
PROBE_CFG = dict(epochs=40, batch_size=512, lr=2e-3, wd_grid=[0.0, 0.01, 0.1, 0.5])
KNN_K_GRID = [10, 50]

In [ ]:
# ---- REUSED VERBATIM from the Q1 notebook (§0 setup); introduced in Q1 ----
import os, math, time, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from scipy import stats
from sklearn.model_selection import train_test_split
from IPython.display import display

warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")
pd.set_option("display.width", 200)

digits = "".join(ch for ch in str(ROLL_NUMBER) if ch.isdigit())
if not digits:
    raise ValueError("Set ROLL_NUMBER in the configuration cell first.")
SEED = int(digits)

def derive_seed(*tags):
    '''32-bit seed derived deterministically from the roll-number seed plus a tag,
    so every random choice in the notebook traces back to SEED.'''
    words = [SEED] + [int.from_bytes(str(t).encode(), "little") for t in tags]
    return int(np.random.SeedSequence(words).generate_state(1)[0])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
os.makedirs(os.path.join(OUT_DIR, "runs"), exist_ok=True)

# Plot style: fixed-order colour-blind-checked categorical palette, recessive grid
PALETTE = ["#257de0", "#f47230", "#00b894", "#f2a600", "#ee7fac", "#008a17", "#6a4ce2", "#ec4647"]
GREY = "#7a7a7a"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.prop_cycle": plt.cycler(color=PALETTE), "axes.grid": True,
    "grid.color": "#e7e7e7", "grid.linewidth": 0.8, "axes.edgecolor": "#c2c2c2",
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "legend.frameon": False, "font.size": 10,
})
def savefig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name))
    plt.show()

def pct(x):
    return f"{100 * x:.1f}%"

def pp(x):
    return f"{100 * x:+.1f} pp"

print(f"Roll number {ROLL_NUMBER!r} -> seed {SEED}")
print(f"device={DEVICE}  AMP={USE_AMP}  torch={torch.__version__}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. The full budget is sized for a GPU; on CPU expect many hours.")

In [ ]:
from torchvision.transforms.functional import gaussian_blur
Q1_RUNS = os.path.join(Q1_OUT_DIR, "runs")
os.makedirs(Q1_RUNS, exist_ok=True)
print(f"Q1 run cache: {os.path.abspath(Q1_RUNS)} -> "
      f"{len([f for f in os.listdir(Q1_RUNS) if f.endswith('.pt')])} cached runs found")

def identity(x):          # defined in Q1's permutation cell; needed before the reused training loop
    return x

def sd(x):
    return float(np.std(x, ddof=1)) if len(x) > 1 else float("nan")

def mean_sd(x, digits=4):
    return f"{np.mean(x):.{digits}f} ± {sd(x):.{digits}f}" if len(x) > 1 else f"{np.mean(x):.{digits}f}"

def psnr(mse):
    return float("inf") if mse <= 0 else 10 * math.log10(1.0 / mse)

def ratio(num, den, min_den=0.01):
    '''num / den, undefined (nan) when the denominator is too small to be meaningful.'''
    return num / den if den > min_den else float("nan")

## 1. Data (reused from Q1 §1)
This uses the same split and normalisation statistics as Q1. The autoencoders use un-normalised [0, 1] copies of the same images.

In [ ]:
# ---- REUSED VERBATIM from the Q1 notebook (§1 data); introduced in Q1 ----
train_full = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True)
X_all = np.asarray(train_full.data)                       # (50000, 32, 32, 3) uint8
y_all = np.asarray(train_full.targets, dtype=np.int64)
CLASSES = list(train_full.classes)

# sklearn needs a 32-bit integer; a roll number with more than 9-10 digits is mapped through SeedSequence
split_state = SEED if SEED < 2**32 else derive_seed("split")
train_idx, val_idx = train_test_split(np.arange(len(y_all)), train_size=N_TRAIN, test_size=N_VAL,
                                      stratify=y_all, random_state=split_state)
np.savez(os.path.join(OUT_DIR, "split_indices.npz"), train_idx=train_idx, val_idx=val_idx, seed=SEED)

split_counts = pd.DataFrame({"train": np.bincount(y_all[train_idx], minlength=10),
                             "validation": np.bincount(y_all[val_idx], minlength=10)}, index=CLASSES)
assert len(np.intersect1d(train_idx, val_idx)) == 0
assert split_counts["train"].sum() == N_TRAIN and split_counts["validation"].sum() == N_VAL
print(f"random_state used for the split: {split_state}")
display(split_counts.T)

# Per-channel normalisation from the training split only
_x = torch.from_numpy(X_all[train_idx]).float().div_(255)
CH_MEAN, CH_STD = _x.mean(dim=(0, 1, 2)), _x.std(dim=(0, 1, 2))
del _x
print("channel mean", CH_MEAN.numpy().round(4), " std", CH_STD.numpy().round(4))

def to_tensor(x_u8):
    '''uint8 NHWC -> normalised float32 NCHW.'''
    x = torch.from_numpy(np.ascontiguousarray(x_u8)).float().div_(255)
    return ((x - CH_MEAN) / CH_STD).permute(0, 3, 1, 2).contiguous()

# The whole split lives on the GPU: no DataLoader overhead, and no augmentation is used anywhere in Q1
Xtr, ytr = to_tensor(X_all[train_idx]).to(DEVICE), torch.from_numpy(y_all[train_idx]).to(DEVICE)
Xva, yva = to_tensor(X_all[val_idx]).to(DEVICE), torch.from_numpy(y_all[val_idx]).to(DEVICE)
print("train", tuple(Xtr.shape), " validation", tuple(Xva.shape))

In [ ]:
q1_split = os.path.join(Q1_OUT_DIR, "split_indices.npz")
if os.path.exists(q1_split):
    s = np.load(q1_split)
    assert np.array_equal(s["train_idx"], train_idx) and np.array_equal(s["val_idx"], val_idx), "split differs from Q1"
    print("Split is identical to the one saved by Q1.")

def to01(x_u8):
    '''uint8 NHWC -> float [0, 1] NCHW (autoencoder space).'''
    return torch.from_numpy(np.ascontiguousarray(x_u8)).float().div_(255).permute(0, 3, 1, 2).contiguous()

Xtr01, Xva01 = to01(X_all[train_idx]).to(DEVICE), to01(X_all[val_idx]).to(DEVICE)
MEAN_D, STD_D = CH_MEAN.to(DEVICE).view(1, 3, 1, 1), CH_STD.to(DEVICE).view(1, 3, 1, 1)

def normalise(x01):
    '''[0, 1] images -> the normalised input space of the Q1 classifier.'''
    return (x01 - MEAN_D) / STD_D
print("autoencoder tensors:", tuple(Xtr01.shape), tuple(Xva01.shape))

## 2. Shared components (reused verbatim from Q1 §2)
These are needed to rebuild and, if its cache is missing, retrain the fixed CNN classifier.

In [ ]:
# ---- REUSED VERBATIM from the Q1 notebook (§2 models and counters); introduced in Q1 ----
class MLP(nn.Module):
    '''Flatten -> [Linear -> BatchNorm1d -> ReLU] x depth -> Linear.'''
    def __init__(self, width, depth=3, in_dim=3 * 32 * 32, n_classes=10):
        super().__init__()
        layers, d = [nn.Flatten()], in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.BatchNorm1d(width), nn.ReLU(inplace=True)]
            d = width
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)
        self.spec = "MLP 3072 -> " + " -> ".join([str(width)] * depth) + " -> 10 (Linear+BN+ReLU per hidden layer)"

    def forward(self, x):
        return self.net(x)


def conv_bn_relu(cin, cout):
    return [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]


class CNN(nn.Module):
    '''Six 3x3 conv layers (BN+ReLU) in three stages with 2x2 max-pooling, global average pooling, linear head.'''
    def __init__(self, ch=(32, 64, 128, 128, 256, 256), n_classes=10):
        super().__init__()
        c1, c2, c3, c4, c5, c6 = ch
        self.features = nn.Sequential(
            *conv_bn_relu(3, c1), *conv_bn_relu(c1, c2), nn.MaxPool2d(2),     # 32x32 -> 16x16
            *conv_bn_relu(c2, c3), *conv_bn_relu(c3, c4), nn.MaxPool2d(2),    # 16x16 -> 8x8
            *conv_bn_relu(c4, c5), *conv_bn_relu(c5, c6), nn.MaxPool2d(2))    # 8x8   -> 4x4
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c6, n_classes))
        self.spec = (f"CNN conv3x3 [{c1},{c2}] pool | [{c3},{c4}] pool | [{c5},{c6}] pool | GAP | "
                     f"Linear {c6}->10 (BN+ReLU after each conv)")

    def forward(self, x):
        return self.head(self.features(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_macs(model, input_shape=(1, 3, 32, 32)):
    '''Multiply-accumulates of conv and linear layers for one image (BN, ReLU and pooling ignored).'''
    macs = 0
    def conv_hook(m, inp, out):
        nonlocal macs
        macs += out.numel() * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
    def lin_hook(m, inp, out):
        nonlocal macs
        macs += out.numel() * m.in_features
    hooks = [m.register_forward_hook(conv_hook) for m in model.modules() if isinstance(m, nn.Conv2d)]
    hooks += [m.register_forward_hook(lin_hook) for m in model.modules() if isinstance(m, nn.Linear)]
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(input_shape, device=next(model.parameters()).device))
    model.train(was_training)
    for h in hooks:
        h.remove()
    return macs


def mlp_param_count(width, depth=3, in_dim=3072, n_classes=10):
    '''Closed form for MLP(width, depth): Linear weights+biases plus BatchNorm scale/shift.'''
    first = in_dim * width + width + 2 * width
    hidden = (depth - 1) * (width * width + width + 2 * width)
    return first + hidden + width * n_classes + n_classes


def width_matching(target, depth=3):
    '''MLP width whose parameter count is closest to `target`.'''
    return min(range(8, 8192), key=lambda w: abs(mlp_param_count(w, depth) - target))

In [ ]:
# ---- REUSED VERBATIM from the Q1 notebook (§2 training loop); introduced in Q1 ----
@torch.no_grad()
def predict_logits(model, X, input_fn=identity, bs=2000):
    model.eval()
    out = []
    for i in range(0, X.shape[0], bs):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(model(input_fn(X[i:i + bs])).float())
    return torch.cat(out)


def loss_acc(logits, y):
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()


def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def df_to_lists(df):
    return {c: df[c].astype(float).tolist() for c in df.columns}


def train_model(model, X, y, Xv, yv, *, epochs, lr, weight_decay, batch_size, schedule, seed,
                input_fn=identity, train_eval_idx=None, step_callback=None, callback_steps=(),
                tag="", print_every=10):
    '''AdamW + (cosine | constant) LR schedule, mixed precision on GPU.
    Per epoch: running train loss/acc, eval-mode loss/acc on a fixed training subset, validation loss/acc.
    Keeps the checkpoint with the best validation accuracy (early-stopping selection uses validation only).'''
    model = model.to(DEVICE)
    n = X.shape[0]
    steps_per_epoch = math.ceil(n / batch_size)
    total_steps = epochs * steps_per_epoch
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if schedule == "cosine":
        lr_lambda = lambda s: 0.5 * (1 + math.cos(math.pi * min(s, total_steps) / total_steps))
    elif schedule == "constant":
        lr_lambda = lambda s: 1.0
    else:
        raise ValueError(schedule)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    gen = torch.Generator().manual_seed(seed)              # data order depends only on the run seed
    callback_steps = set(callback_steps)
    rows, best, step, t0 = [], dict(val_acc=-1.0, epoch=0, state=None), 0, time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(n, generator=gen).to(DEVICE)
        loss_sum = torch.zeros((), device=DEVICE)
        n_correct = torch.zeros((), device=DEVICE)
        for i in range(0, n, batch_size):
            idx = order[i:i + batch_size]
            xb, yb = input_fn(X[idx]), y[idx]
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.detach().float() * len(idx)
            n_correct += (logits.argmax(1) == yb).sum()
            step += 1
            if step_callback is not None and step in callback_steps:
                step_callback(model, step / steps_per_epoch)
                model.train()

        row = dict(epoch=epoch, lr=sched.get_last_lr()[0],
                   train_loss_running=loss_sum.item() / n, train_acc_running=n_correct.item() / n)
        row["val_loss"], row["val_acc"] = loss_acc(predict_logits(model, Xv, input_fn), yv)
        if train_eval_idx is not None:
            row["train_loss"], row["train_acc"] = loss_acc(
                predict_logits(model, X[train_eval_idx], input_fn), y[train_eval_idx])
        rows.append(row)
        if row["val_acc"] > best["val_acc"]:
            best = dict(val_acc=row["val_acc"], epoch=epoch, state=cpu_state(model))
        if print_every and (epoch == 1 or epoch % print_every == 0 or epoch == epochs):
            extra = f" train(eval) {row['train_acc']:.3f}" if "train_acc" in row else ""
            print(f"  [{tag}] ep {epoch:3d}/{epochs} loss {row['train_loss_running']:.3f} "
                  f"train(run) {row['train_acc_running']:.3f}{extra} val {row['val_acc']:.3f} "
                  f"({time.time() - t0:.0f}s)")

    return dict(history=pd.DataFrame(rows), best_epoch=best["epoch"], best_val_acc=best["val_acc"],
                best_state=best["state"], final_state=cpu_state(model), seconds=time.time() - t0,
                steps=step)

## 3. The fixed CNN classifier (reused from Q1(a))
The classifier is the **CNN from Q1(a)** (also the CNN compared in Q2(a)), trained on clean images. It uses seed #0 at the
learning rate Q1 selected on validation, with its best-validation checkpoint. It is loaded from the Q1 run cache. If the cache is missing, the Q1(a) LR sweep for this model is
re-run with identical code, seeds and settings. The classifier is frozen: no fine-tuning anywhere in Q3.

In [ ]:
CACHE_TAG = hashlib.md5(json.dumps([SEED, N_TRAIN, N_VAL, CFG_A, TRAIN_EVAL_SUBSET],
                                   sort_keys=True).encode()).hexdigest()[:8]           # identical to Q1
TRAIN_EVAL_IDX = torch.from_numpy(np.random.default_rng(derive_seed("train-eval-subset")).choice(
    N_TRAIN, TRAIN_EVAL_SUBSET, replace=False)).to(DEVICE)                             # identical to Q1
CLF_SOURCE = {}

def q1_cnn_run(lr, seed_idx=0):
    path = os.path.join(Q1_RUNS, f"q1a_{CACHE_TAG}_CNN_lr{lr:g}_s{seed_idx}.pt")
    if os.path.exists(path):
        CLF_SOURCE[lr] = "reused: loaded from Q1(a) run cache"
        return torch.load(path, map_location="cpu")
    seed = derive_seed("q1a-run", seed_idx)
    torch.manual_seed(seed)
    model = CNN()
    res = train_model(model, Xtr, ytr, Xva, yva, lr=lr, seed=seed, input_fn=identity,
                      train_eval_idx=TRAIN_EVAL_IDX, tag=f"Q1(a) CNN lr={lr:g} seed#{seed_idx}", **CFG_A)
    out = dict(name="CNN", lr=lr, seed_idx=seed_idx, history=df_to_lists(res["history"]),
               best_epoch=int(res["best_epoch"]), best_val_acc=float(res["best_val_acc"]),
               best_state=res["best_state"], seconds=float(res["seconds"]))
    torch.save(out, path)
    CLF_SOURCE[lr] = "retrained here with the Q1(a) configuration (Q1 cache absent)"
    return out

clf_sweep = {lr: q1_cnn_run(lr) for lr in LR_GRID}
CLF_LR = max(LR_GRID, key=lambda l: clf_sweep[l]["best_val_acc"])     # Q1's selection rule
CLF = CNN().to(DEVICE)
CLF.load_state_dict(clf_sweep[CLF_LR]["best_state"])
CLF.eval()
for p in CLF.parameters():
    p.requires_grad_(False)

def classify(X01):
    return predict_logits(CLF, X01, input_fn=normalise)

clf_val_acc = loss_acc(classify(Xva01), yva)[1]
print(f"Fixed classifier: Q1(a) CNN, LR {CLF_LR:g}, seed #0, {sum(p.numel() for p in CLF.parameters()):,} parameters "
      f"({CLF_SOURCE[CLF_LR]}). Validation accuracy on clean images: {pct(clf_val_acc)}")

## 4(a). Three autoencoders that differ only in bottleneck capacity

**Architecture (identical except for *d*).**
* **Encoder:** conv3×3(3→32), then three stride-2 conv4×4 layers (32→64→128→256, giving 32→16→8→4 px), each followed by BatchNorm and ReLU. The 256×4×4 = 4096 features are flattened and passed through a **linear bottleneck to *d* units** (no non-linearity, so *z* ∈ ℝ^d is the representation).
* **Decoder:** the mirror image. A linear layer maps *d* back to 4096, followed by ReLU, a reshape to 256×4×4, three stride-2 transposed conv4×4 layers (256→128→64→32) with BN and ReLU, and a conv3×3(32→3) with a sigmoid output.
* **Bottleneck sizes:** *d* = 16, 128 and 1024. Each is 8× the previous, giving 192×, 24× and 3× compression of the 3072 input values, so the three are clearly separated. Only the two linear layers next to the bottleneck change size, so the parameter difference comes from capacity alone.

**Training setup (identical for every *d*).** MSE between reconstruction and input in [0, 1] pixel space, AdamW (LR 1e-3,
weight decay 1e-4), cosine schedule, 30 epochs, batch 128, no augmentation, and no labels. Two seeds per size; seed *k* uses
the same initialisation stream and batch order for every *d*. The checkpoint kept is the one with the lowest
**validation** MSE. Adam at 1e-3 is a standard, robust setting for autoencoders, so it is fixed rather than tuned. The
convergence column below checks that no size is under-trained (validation MSE changes by less than about 2% over the last 5 epochs).

**Selection rule (if reconstruction error were the only criterion; fixed in advance).** Choose the bottleneck with
the lowest mean validation MSE. If a smaller bottleneck is within 2 pooled seed-sd of the best, treat the two as tied
and take the smaller one, because the reconstruction criterion cannot tell them apart.

In [ ]:
class ConvAE(nn.Module):
    '''Convolutional autoencoder; everything except the bottleneck width d is fixed.'''
    def __init__(self, d):
        super().__init__()
        self.d = d
        def down(cin, cout):
            return [nn.Conv2d(cin, cout, 4, stride=2, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        def up(cin, cout):
            return [nn.ConvTranspose2d(cin, cout, 4, stride=2, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),     # 32x32
            *down(32, 64), *down(64, 128), *down(128, 256),                                 # 16, 8, 4
            nn.Flatten(), nn.Linear(256 * 4 * 4, d))                                        # bottleneck z
        self.decoder = nn.Sequential(
            nn.Linear(d, 256 * 4 * 4), nn.ReLU(inplace=True), nn.Unflatten(1, (256, 4, 4)),
            *up(256, 128), *up(128, 64), *up(64, 32),
            nn.Conv2d(32, 3, 3, padding=1), nn.Sigmoid())
        self.spec = (f"enc conv3x3 32 | conv4x4/2 64,128,256 | Linear 4096->{d} || "
                     f"dec Linear {d}->4096 | convT4x4/2 128,64,32 | conv3x3 3 + sigmoid")

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.decoder(self.encoder(x))


@torch.no_grad()
def reconstruct(model, X, bs=1000):
    model.eval()
    out = []
    for i in range(0, X.shape[0], bs):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out.append(model(X[i:i + bs]).float())
    return torch.cat(out)


def mse_per_image(A, B):
    return ((A - B) ** 2).flatten(1).mean(1)


def train_autoencoder(model, X, Xv_in, Xv_target, *, epochs, lr, weight_decay, batch_size, seed,
                      corrupt=None, train_eval_idx=None, tag="", print_every=10):
    '''MSE autoencoder training on [0, 1] images. `corrupt` (DAE only) maps a clean batch to a corrupted input;
    the target is always the clean batch. Keeps the checkpoint with the lowest validation MSE.'''
    model = model.to(DEVICE)
    n = X.shape[0]
    total = epochs * math.ceil(n / batch_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    gen = torch.Generator().manual_seed(seed)
    rows, best, t0 = [], dict(val_mse=float("inf"), epoch=0, state=None), time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(n, generator=gen).to(DEVICE)
        loss_sum = torch.zeros((), device=DEVICE)
        for i in range(0, n, batch_size):
            x = X[order[i:i + batch_size]]
            inp = corrupt(x) if corrupt is not None else x
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(inp)
            loss = F.mse_loss(out.float(), x)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            loss_sum += loss.detach() * len(x)
        row = dict(epoch=epoch, train_mse_running=loss_sum.item() / n,
                   val_mse=float(mse_per_image(reconstruct(model, Xv_in), Xv_target).mean()))
        if train_eval_idx is not None:
            xt = X[train_eval_idx]
            xin = corrupt(xt) if corrupt is not None else xt
            row["train_mse"] = float(mse_per_image(reconstruct(model, xin), xt).mean())
        rows.append(row)
        if row["val_mse"] < best["val_mse"]:
            best = dict(val_mse=row["val_mse"], epoch=epoch, state=cpu_state(model))
        if print_every and (epoch == 1 or epoch % print_every == 0 or epoch == epochs):
            print(f"  [{tag}] ep {epoch:3d}/{epochs} train {row['train_mse_running']:.5f} "
                  f"val {row['val_mse']:.5f} ({time.time() - t0:.0f}s)")
    return dict(history=pd.DataFrame(rows), best_epoch=best["epoch"], best_val_mse=best["val_mse"],
                best_state=best["state"], seconds=time.time() - t0)


# ---- Corruption helpers (used from part (b) onwards; defined here because run_ae needs them) ----
def add_noise_fixed(X01, sigma, split, chunk=5000):
    '''Deterministic noise for evaluation: the same epsilon per image for a split (seeded), scaled by sigma.'''
    g = torch.Generator().manual_seed(derive_seed("gaussian-noise", split))
    out = torch.empty_like(X01)
    for i in range(0, X01.shape[0], chunk):
        eps = torch.randn(X01[i:i + chunk].shape, generator=g).to(X01.device)
        out[i:i + chunk] = (X01[i:i + chunk] + sigma * eps).clamp_(0, 1)
    return out

def noise_fn(sigma):
    '''Training-time corruption: fresh noise every time a batch is drawn.'''
    return lambda x: (x + sigma * torch.randn_like(x)).clamp_(0, 1)

XVA_NOISY = add_noise_fixed(Xva01, NOISE_SIGMA, "val")

AE_TAG = hashlib.md5(json.dumps([SEED, N_TRAIN, N_VAL, CFG_AE, TRAIN_EVAL_SUBSET],
                                sort_keys=True).encode()).hexdigest()[:8]

def run_ae(kind, d, seed_idx):
    '''kind = "ae" (clean -> clean) or "dae" (noisy -> clean). Same seed stream for every d and for AE vs DAE.'''
    tag = AE_TAG + (f"_sigma{NOISE_SIGMA:g}" if kind == "dae" else "")
    path = os.path.join(OUT_DIR, "runs", f"q3_{tag}_{kind}_d{d}_s{seed_idx}.pt")
    if os.path.exists(path):
        out = torch.load(path, map_location="cpu")
    else:
        seed = derive_seed("q3-ae", seed_idx)
        torch.manual_seed(seed)
        model = ConvAE(d)
        res = train_autoencoder(model, Xtr01, Xva01 if kind == "ae" else XVA_NOISY, Xva01, seed=seed,
                                corrupt=noise_fn(NOISE_SIGMA) if kind == "dae" else None,
                                train_eval_idx=TRAIN_EVAL_IDX, tag=f"{kind.upper()} d={d} seed#{seed_idx}", **CFG_AE)
        out = dict(kind=kind, d=d, seed_idx=seed_idx, history=df_to_lists(res["history"]),
                   best_epoch=int(res["best_epoch"]), best_val_mse=float(res["best_val_mse"]),
                   best_state=res["best_state"], seconds=float(res["seconds"]))
        torch.save(out, path)
    out["history"] = pd.DataFrame(out["history"])
    return out

def load_ae(run):
    m = ConvAE(run["d"]).to(DEVICE)
    m.load_state_dict(run["best_state"])
    return m.eval()

In [ ]:
T0 = time.time()
AE_RUNS = {d: [run_ae("ae", d, s) for s in range(N_SEEDS_AE)] for d in BOTTLENECKS}

rows = []
for d in BOTTLENECKS:
    m, runs = ConvAE(d), AE_RUNS[d]
    v = [r["best_val_mse"] for r in runs]
    last = [r["history"]["val_mse"].values for r in runs]
    rows.append(dict(bottleneck_d=d, compression=f"{3072 / d:.0f}x", parameters=count_params(m),
                     encoder_params=count_params(m.encoder), decoder_params=count_params(m.decoder),
                     bottleneck_layer_params=count_params(m.encoder[-1]) + count_params(m.decoder[0]),
                     val_mse=mean_sd(v, 5), val_mse_mean=float(np.mean(v)), val_psnr_db=psnr(float(np.mean(v))),
                     best_epoch=np.mean([r["best_epoch"] for r in runs]),
                     train_mse_final=np.mean([r["history"]["train_mse"].iloc[-1] for r in runs]),
                     val_change_last5=np.mean([(h[-min(6, len(h))] - h[-1]) / h[-1] for h in last])))
TABLE_A = pd.DataFrame(rows).set_index("bottleneck_d")
TABLE_A.to_csv(os.path.join(OUT_DIR, "table_q3a_autoencoders_validation.csv"))
print(f"Table Q3-1: autoencoders (validation; mean ± sd over {N_SEEDS_AE} seeds). AE training this session: "
      f"{(time.time() - T0) / 60:.1f} min")
with pd.option_context("display.max_colwidth", 40):
    display(TABLE_A.drop(columns="val_mse_mean").round(5))
print("Architecture:", ConvAE(BOTTLENECKS[0]).spec.replace(str(BOTTLENECKS[0]), "d"))

# ---- Selection on validation reconstruction error only ----
means = {d: TABLE_A.loc[d, "val_mse_mean"] for d in BOTTLENECKS}
pooled_sd = (float(np.sqrt(np.mean([np.var([r["best_val_mse"] for r in AE_RUNS[d]], ddof=1) for d in BOTTLENECKS])))
             if N_SEEDS_AE > 1 else float("nan"))
best_d = min(means, key=means.get)
tied = [d for d in BOTTLENECKS if not np.isnan(pooled_sd) and means[d] - means[best_d] <= 2 * pooled_sd]
SELECTED_D = min(tied) if tied else best_d
order = sorted(BOTTLENECKS)
steps = [f"d {a}->{b}: {100 * (means[a] - means[b]) / means[a]:.1f}% lower MSE" for a, b in zip(order[:-1], order[1:])]
second = sorted(means, key=means.get)[1]
SELECTION_TEXT = (
    f"Selected AE-{SELECTED_D}. It has the lowest validation MSE ({means[SELECTED_D]:.5f}, PSNR "
    f"{psnr(means[SELECTED_D]):.2f} dB); the next best (AE-{second}) is {means[second]:.5f}, a gap of "
    f"{(means[second] - means[best_d]) / pooled_sd:.1f} pooled seed-sd, so the ranking is not seed noise. "
    if SELECTED_D == best_d and not np.isnan(pooled_sd) else
    f"Selected AE-{SELECTED_D} (lowest validation MSE {means[best_d]:.5f} for AE-{best_d}; tie rule applied: {tied}). ")
SELECTION_TEXT += ("A wider bottleneck discards less information, and reconstruction error is the only criterion, so "
                   "nothing here penalises the extra capacity. Validation MSE change per 8x step in d: " +
                   "; ".join(steps) + ".")
print(SELECTION_TEXT)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for k, d in enumerate(BOTTLENECKS):
    H = np.stack([r["history"]["val_mse"].values for r in AE_RUNS[d]])
    Ht = np.stack([r["history"]["train_mse"].values for r in AE_RUNS[d]])
    ep = np.arange(1, H.shape[1] + 1)
    axes[0].plot(ep, H.mean(0), color=PALETTE[k], label=f"d = {d} validation")
    axes[0].plot(ep, Ht.mean(0), color=PALETTE[k], ls=":", lw=1.3, label=f"d = {d} train (eval mode)")
axes[0].set_yscale("log"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE per pixel (log scale)")
axes[0].set_title("Reconstruction error during training", fontsize=10)
axes[0].legend(fontsize=7, ncol=2)
axes[1].errorbar(BOTTLENECKS, [means[d] for d in BOTTLENECKS],
                 yerr=[sd([r["best_val_mse"] for r in AE_RUNS[d]]) if N_SEEDS_AE > 1 else 0 for d in BOTTLENECKS],
                 marker="o", ms=7, capsize=3, color=PALETTE[0])
axes[1].scatter([SELECTED_D], [means[SELECTED_D]], s=160, facecolors="none", edgecolors=PALETTE[7], lw=2,
                label="selected")
axes[1].set_xscale("log", base=2); axes[1].set_yscale("log")
axes[1].set_xlabel("bottleneck width d (log scale)"); axes[1].set_ylabel("best validation MSE (log scale)")
axes[1].set_title("Validation reconstruction error vs capacity", fontsize=10)
axes[1].legend(fontsize=8)
fig.suptitle("Fig. Q3-1: autoencoders differing only in bottleneck capacity", fontsize=10)
fig.tight_layout()
savefig(fig, "fig_q3_1_ae_training.png")

# Qualitative reconstructions of validation images
show_idx = [int(np.where(y_all[val_idx] == c)[0][0]) for c in range(8)]
fig, axes = plt.subplots(1 + len(BOTTLENECKS), 8, figsize=(9.5, 1.3 * (1 + len(BOTTLENECKS))))
rows_img = [("original", Xva01[show_idx])] + [(f"AE-{d}", reconstruct(load_ae(AE_RUNS[d][0]), Xva01[show_idx]))
                                              for d in BOTTLENECKS]
for i, (lab, imgs) in enumerate(rows_img):
    for j in range(8):
        ax = axes[i, j]
        ax.imshow(imgs[j].permute(1, 2, 0).cpu().numpy()); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if j == 0:
            ax.set_ylabel(lab, fontsize=8)
fig.suptitle("Fig. Q3-2: validation images and their reconstructions (seed #0)", fontsize=10)
fig.tight_layout()
savefig(fig, "fig_q3_2_reconstructions.png")

## 4(b). Denoising: does it give the classifier back what it needs?

**Corruption: additive Gaussian noise, σ = 0.2** on [0, 1] pixels, clipped to [0, 1]. Per-pixel noise of this size badly hurts a
classifier trained on clean images, but it is a corruption where **reconstruction can plausibly help**. The noise is independent
from pixel to pixel and high-frequency, while natural images are dominated by smooth, spatially correlated structure, so
a model that has learned image statistics can separate the two. By contrast, a large occlusion removes information
outright, and any reconstruction there would be invention.

**Denoising training protocol (no labels used anywhere).**
| Item | Setting |
|---|---|
| Architecture | identical to the **selected clean AE from (a)** (same *d*), so the only difference between the two is the training input |
| Input / target | input = clip(x + σ·ε, 0, 1), ε ~ N(0, I), σ = 0.2; target = the clean image x |
| Noise sampling | ε is redrawn every time a training image is used, so the model never sees the same noisy copy twice |
| Loss / optimiser / budget | MSE to the clean image; AdamW 1e-3, wd 1e-4, cosine, 30 epochs, batch 128 (same as (a)) |
| Checkpoint selection | lowest validation MSE between DAE(noisy validation image, fixed noise) and the clean validation image |
| Seeds | 2, sharing initialisation and batch order with the clean AE's seeds |

**Comparison, all with the same frozen CNN classifier.** Clean images (reference), corrupted images, a non-learned
Gaussian-blur denoiser (blur σ chosen on **validation MSE**, so also label-free), the **clean AE** reconstructing the corrupted
image (baseline reused from (a)), and the **DAE** reconstructing the corrupted image. Two ceilings are also measured: the clean
AE and the DAE applied to *clean* images. These show how much accuracy the classifier loses to reconstruction blur alone.

**Decision rule (fixed in advance).** Recovered fraction F = (acc(DAE(x̃)) − acc(x̃)) / (acc(x) − acc(x̃)), where x is the clean
image and x̃ the corrupted one, measured on the test set.
*Recovers* if F ≥ 0.5 and the 95% bootstrap CI of acc(DAE(x̃)) − acc(x̃) excludes 0. *Partially* if the CI excludes 0 but F < 0.5.
*Does not recover* otherwise. Comparing with the ceiling acc(AE(x)) separates lost information from the classifier's
dislike of smooth reconstructions.

In [ ]:
DAE_RUNS = [run_ae("dae", SELECTED_D, s) for s in range(N_SEEDS_AE)]
AE_SEL_RUNS = AE_RUNS[SELECTED_D]

blur_mse = {}
for s_ in BLUR_SIGMA_GRID:
    k = 2 * math.ceil(3 * s_) + 1
    blur_mse[s_] = float(mse_per_image(gaussian_blur(XVA_NOISY, [k, k], [s_, s_]), Xva01).mean())
BLUR_SIGMA = min(blur_mse, key=blur_mse.get)

def blur(X, s=None):
    s = BLUR_SIGMA if s is None else s
    k = 2 * math.ceil(3 * s) + 1
    return torch.cat([gaussian_blur(X[i:i + 5000], [k, k], [s, s]) for i in range(0, X.shape[0], 5000)])

print(f"Blur baseline: sigma = {BLUR_SIGMA} chosen on validation MSE "
      f"({', '.join(f'{k}: {v:.5f}' for k, v in blur_mse.items())})")
print(f"DAE (d = {SELECTED_D}) best validation MSE (noisy -> clean): "
      f"{mean_sd([r['best_val_mse'] for r in DAE_RUNS], 5)}; noisy input itself: "
      f"{float(mse_per_image(XVA_NOISY, Xva01).mean()):.5f}")

def condition_images(X_clean, X_noisy):
    '''All (b) conditions for one split. Reconstructions are lists over seeds.'''
    ae = [load_ae(r) for r in AE_SEL_RUNS]
    dae = [load_ae(r) for r in DAE_RUNS]
    return {"clean images": [X_clean],
            f"corrupted (sigma={NOISE_SIGMA:g})": [X_noisy],
            f"Gaussian blur (s={BLUR_SIGMA:g}) of corrupted": [blur(X_noisy)],
            f"clean AE-{SELECTED_D} of corrupted": [reconstruct(m, X_noisy) for m in ae],
            f"DAE-{SELECTED_D} of corrupted": [reconstruct(m, X_noisy) for m in dae],
            f"ceiling: clean AE-{SELECTED_D} of clean": [reconstruct(m, X_clean) for m in ae],
            f"ceiling: DAE-{SELECTED_D} of clean": [reconstruct(m, X_clean) for m in dae]}

def condition_table(imgs, X_clean, y):
    rows = []
    for name, lst in imgs.items():
        accs = [loss_acc(classify(Z), y)[1] for Z in lst]
        confs = [float(classify(Z).softmax(1).max(1).values.mean()) for Z in lst]
        mses = [float(mse_per_image(Z, X_clean).mean()) for Z in lst]
        rows.append(dict(condition=name, classifier_acc=mean_sd(accs), acc_mean=np.mean(accs),
                         mean_confidence=np.mean(confs), mse_to_clean=np.mean(mses), psnr_db=psnr(np.mean(mses))))
    return pd.DataFrame(rows).set_index("condition")

VAL_B = condition_table(condition_images(Xva01, XVA_NOISY), Xva01, yva)
print("Validation view of part (b) (nothing is selected from these classifier numbers):")
display(VAL_B.drop(columns="acc_mean").round(4))

idx8 = [int(np.where(y_all[val_idx] == c)[0][0]) for c in range(8)]
imgs_v = condition_images(Xva01[idx8], XVA_NOISY[idx8])
keys = list(imgs_v)[:5]
fig, axes = plt.subplots(len(keys), 8, figsize=(9.5, 1.35 * len(keys)))
for i, kname in enumerate(keys):
    for j in range(8):
        ax = axes[i, j]
        ax.imshow(imgs_v[kname][0][j].permute(1, 2, 0).cpu().numpy()); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if j == 0:
            ax.set_ylabel(kname.replace(" of corrupted", "").replace("Gaussian ", ""), fontsize=7)
fig.suptitle("Fig. Q3-3: validation images: clean, corrupted, and three ways of cleaning them", fontsize=10)
fig.tight_layout()
savefig(fig, "fig_q3_3_denoising_examples.png")

## 4(c). How much class information is in the representations, and does it track reconstruction quality?

**Representations (frozen; no encoder is fine-tuned).** For each model, the representation is the bottleneck vector *z* = encoder(x) ∈ ℝ^d, computed in eval mode:
* the three clean AEs from (a): AE-16, AE-128, AE-1024 (2 seeds each), reused;
* the DAE from (b) (same *d* as the selected AE), reused;
* **reference points:** raw pixels (3072-d, which is a perfect "reconstruction"), and **randomly initialised, untrained** encoders of each width. The random encoders show how much class information the architecture gives by itself, before any reconstruction training.

**Probes (only these see labels).**
* **Linear probe:** features standardised with training-split statistics, then multinomial logistic regression (AdamW, 40 epochs, batch 512), with weight decay chosen from {0, 0.01, 0.1, 0.5} on validation accuracy.
* **k-NN probe:** cosine similarity on standardised features, k ∈ {10, 50} chosen on validation. It needs no training and so checks that the linear probe is not the bottleneck.

Probes are trained on the 45k training split, selected on validation, and reported on the **test** set. A supplementary
check links this to (b): probes trained and tested on representations of **noisy** images (fixed noise, σ = 0.2) for the selected clean AE and for the DAE.

**Reconstruction quality** is the test MSE on clean images (lower is better), the same measure for every model.

**Decision rule (fixed in advance).** The two quantities *agree* if, across the four trained encoders (AE-16, AE-128, AE-1024
and the DAE), the Spearman correlation between reconstruction quality (−MSE) and linear-probe test accuracy is ≥ 0.8. They *disagree*
if it is ≤ 0, and agree *partially* in between. The raw-pixel and random-encoder reference points are used to check whether any agreement holds
outside the trained models.

In [ ]:
def features(model, X01, bs=1000):
    if model is None:
        return X01.flatten(1).clone()
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, X01.shape[0], bs):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                out.append(model.encode(X01[i:i + bs]).float())
    return torch.cat(out)

def train_linear_probe(Ftr, y_tr, Fva, y_va, seed):
    mu, sdv = Ftr.mean(0), Ftr.std(0) + 1e-6
    Ztr, Zva = (Ftr - mu) / sdv, (Fva - mu) / sdv
    n, dim = Ztr.shape
    spe = math.ceil(n / PROBE_CFG["batch_size"])
    best = dict(val_acc=-1.0)
    for wd in PROBE_CFG["wd_grid"]:
        torch.manual_seed(seed)
        lin = nn.Linear(dim, 10).to(DEVICE)
        opt = torch.optim.AdamW(lin.parameters(), lr=PROBE_CFG["lr"], weight_decay=wd)
        total = PROBE_CFG["epochs"] * spe
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(s, total) / total)))
        gen = torch.Generator().manual_seed(seed)
        for _ in range(PROBE_CFG["epochs"]):
            order = torch.randperm(n, generator=gen).to(DEVICE)
            for i in range(0, n, PROBE_CFG["batch_size"]):
                idx = order[i:i + PROBE_CFG["batch_size"]]
                loss = F.cross_entropy(lin(Ztr[idx]), y_tr[idx])
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                sched.step()
        with torch.no_grad():
            va = (lin(Zva).argmax(1) == y_va).float().mean().item()
        if va > best["val_acc"]:
            best = dict(val_acc=va, wd=wd, W=lin.weight.detach().clone(), b=lin.bias.detach().clone())
    best.update(mu=mu, sd=sdv)
    return best

def knn_predict(Ftr_n, y_tr, Fq_n, k, chunk=1000):
    preds = []
    for i in range(0, Fq_n.shape[0], chunk):
        sims = Fq_n[i:i + chunk] @ Ftr_n.T
        val, idx = sims.topk(k, dim=1)
        votes = torch.zeros(len(idx), 10, device=DEVICE).scatter_add_(1, y_tr[idx], 1 + 1e-3 * val)
        preds.append(votes.argmax(1))
    return torch.cat(preds)

def knn_normalise(Fx, mu, sdv):
    Z = (Fx - mu) / sdv
    return Z / Z.norm(dim=1, keepdim=True).clamp_min(1e-6)

# ---- encoders to probe ----
ENC_SPECS = [("raw pixels", "raw", 3072, None)]
ENC_SPECS += [(f"random encoder d={d}", "random", d, None) for d in BOTTLENECKS]
ENC_SPECS += [(f"AE-{d}", "ae", d, s) for d in BOTTLENECKS for s in range(N_SEEDS_AE)]
ENC_SPECS += [(f"DAE-{SELECTED_D}", "dae", SELECTED_D, s) for s in range(N_SEEDS_AE)]

def get_encoder(kind, d, s):
    if kind == "raw":
        return None
    if kind == "random":
        torch.manual_seed(derive_seed("q3-random-encoder", d))
        return ConvAE(d).to(DEVICE).eval()
    return load_ae(AE_RUNS[d][s] if kind == "ae" else DAE_RUNS[s])

XTR_NOISY = add_noise_fixed(Xtr01, NOISE_SIGMA, "train")
PROBES = []
t0 = time.time()
for name, kind, d, s in ENC_SPECS:
    enc = get_encoder(kind, d, s)
    inputs = [("clean", Xtr01, Xva01)]
    if (kind == "ae" and d == SELECTED_D) or kind == "dae":
        inputs.append(("noisy", XTR_NOISY, XVA_NOISY))
    for inp_name, Xa, Xb in inputs:
        Ftr, Fva = features(enc, Xa), features(enc, Xb)
        lp = train_linear_probe(Ftr, ytr, Fva, yva, seed=derive_seed("probe", name, s or 0, inp_name))
        Ftr_n, Fva_n = knn_normalise(Ftr, lp["mu"], lp["sd"]), knn_normalise(Fva, lp["mu"], lp["sd"])
        knn_val = {k: (knn_predict(Ftr_n, ytr, Fva_n, k) == yva).float().mean().item() for k in KNN_K_GRID}
        k_sel = max(knn_val, key=knn_val.get)
        PROBES.append(dict(name=name, kind=kind, d=d, seed=s, input=inp_name, lin=lp, k=k_sel,
                           val_linear=lp["val_acc"], val_knn=knn_val[k_sel]))
        del Ftr, Fva, Ftr_n, Fva_n
print(f"{len(PROBES)} probes trained in {(time.time() - t0) / 60:.1f} min")
VAL_C = (pd.DataFrame([dict(representation=p["name"], input=p["input"], d=p["d"], val_linear_probe=p["val_linear"],
                            val_knn=p["val_knn"], chosen_wd=p["lin"]["wd"], chosen_k=p["k"]) for p in PROBES])
         .groupby(["representation", "input"], sort=False).mean(numeric_only=True))
print("Validation probe accuracy (selection only; test results in Section 5):")
display(VAL_C.round(4))

## 5. Final test-set evaluation (once, after every choice is fixed)
The bottleneck, checkpoints, blur σ and probe hyperparameters are all fixed at this point. The official test set is loaded for the first
time. Test noise is drawn once with a fixed seed, so every method sees exactly the same corrupted test images.

In [ ]:
test_raw = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True)
Xte01 = to01(np.asarray(test_raw.data)).to(DEVICE)
yte = torch.from_numpy(np.asarray(test_raw.targets, dtype=np.int64)).to(DEVICE)
XTE_NOISY = add_noise_fixed(Xte01, NOISE_SIGMA, "test")

# ---------------- (a) test reconstruction error ----------------
rows = []
TEST_MSE = {}
for d in BOTTLENECKS:
    v = [float(mse_per_image(reconstruct(load_ae(r), Xte01), Xte01).mean()) for r in AE_RUNS[d]]
    TEST_MSE[f"AE-{d}"] = v
    rows.append(dict(bottleneck_d=d, parameters=count_params(ConvAE(d)), selected=(d == SELECTED_D),
                     val_mse=TABLE_A.loc[d, "val_mse"], test_mse=mean_sd(v, 5), test_psnr_db=psnr(np.mean(v))))
TEST_A = pd.DataFrame(rows).set_index("bottleneck_d")
TEST_A.to_csv(os.path.join(OUT_DIR, "table_q3a_test_reconstruction.csv"))
print(f"Table Q3-2: TEST reconstruction error (MSE per pixel, [0,1] scale). Selected model: AE-{SELECTED_D}")
display(TEST_A.round(5))

# ---------------- (b) classification on corrupted / reconstructed test images ----------------
imgs_te = condition_images(Xte01, XTE_NOISY)
TEST_B = condition_table(imgs_te, Xte01, yte)
TEST_B.to_csv(os.path.join(OUT_DIR, "table_q3b_test_classification.csv"))
print(f"Table Q3-3: TEST classification by the fixed Q1(a) CNN (mean ± sd over {N_SEEDS_AE} autoencoder seeds)")
display(TEST_B.drop(columns="acc_mean").round(4))

names = list(imgs_te)
acc = {n: TEST_B.loc[n, "acc_mean"] for n in names}
a_clean, a_noisy = acc[names[0]], acc[names[1]]
a_blur, a_ae, a_dae, a_ceil_ae, a_ceil_dae = (acc[names[i]] for i in (2, 3, 4, 5, 6))
F_rec = ratio(a_dae - a_noisy, a_clean - a_noisy)
F_ceil = ratio(a_dae - a_noisy, a_ceil_ae - a_noisy)

def correct_matrix(lst):
    return np.stack([(classify(Z).argmax(1) == yte).cpu().numpy() for Z in lst]).mean(0)

def boot_ci(diff, n_boot=2000):
    rng = np.random.default_rng(derive_seed("bootstrap"))
    idx = rng.integers(0, len(diff), (n_boot, len(diff)))
    m = diff[idx].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

c_noisy, c_dae, c_ae = correct_matrix(imgs_te[names[1]]), correct_matrix(imgs_te[names[4]]), correct_matrix(imgs_te[names[3]])
c_blur = correct_matrix(imgs_te[names[2]])
CI = {"DAE - corrupted": boot_ci(c_dae - c_noisy), "DAE - clean AE": boot_ci(c_dae - c_ae),
      "DAE - blur": boot_ci(c_dae - c_blur)}
B_DECISION = ("RECOVERS" if (not np.isnan(F_rec) and F_rec >= 0.5 and CI["DAE - corrupted"][0] > 0) else
              "PARTIALLY RECOVERS" if CI["DAE - corrupted"][0] > 0 else "DOES NOT RECOVER")
display(pd.DataFrame({"difference in accuracy": [pp(a_dae - a_noisy), pp(a_dae - a_ae), pp(a_dae - a_blur)],
                      "95% bootstrap CI": [f"[{pp(lo)}, {pp(hi)}]" for lo, hi in CI.values()]},
                     index=list(CI)))
print(f"Recovered fraction F = {F_rec:.2f} of the clean-vs-corrupted gap; {F_ceil:.2f} of the gap up to the "
      f"reconstruction ceiling (clean AE on clean images, {pct(a_ceil_ae)}). Decision: denoising {B_DECISION} "
      "the information the classifier needs.")

# Supplementary: severity sweep (DAE trained at sigma = NOISE_SIGMA only)
sweep_rows = []
ae_m, dae_m = [load_ae(r) for r in AE_SEL_RUNS], [load_ae(r) for r in DAE_RUNS]
for s_ in SEVERITY_SWEEP:
    Xn = add_noise_fixed(Xte01, s_, "test")
    sweep_rows.append(dict(sigma=s_, corrupted=loss_acc(classify(Xn), yte)[1], blur=loss_acc(classify(blur(Xn)), yte)[1],
                           clean_AE=np.mean([loss_acc(classify(reconstruct(m, Xn)), yte)[1] for m in ae_m]),
                           DAE=np.mean([loss_acc(classify(reconstruct(m, Xn)), yte)[1] for m in dae_m])))
SWEEP = pd.DataFrame(sweep_rows).set_index("sigma")
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for k, (col, lab) in enumerate([("corrupted", "corrupted input"), ("blur", f"Gaussian blur (s={BLUR_SIGMA:g})"),
                                ("clean_AE", f"clean AE-{SELECTED_D}"), ("DAE", f"DAE-{SELECTED_D}")]):
    ax.plot(SWEEP.index, SWEEP[col], marker="o", ms=6, color=PALETTE[k], label=lab)
ax.axhline(a_clean, color=GREY, ls=":", lw=1.2)
ax.text(SWEEP.index.min(), a_clean + 0.01, "clean test images", color=GREY, fontsize=7)
ax.axvline(NOISE_SIGMA, color=GREY, lw=0.8)
ax.set_ylim(0, 1); ax.set_xlabel("noise sigma at test time (DAE trained at %.2g)" % NOISE_SIGMA)
ax.set_ylabel("test accuracy of the fixed CNN")
ax.set_title("Fig. Q3-4 (supplementary): accuracy vs noise severity", fontsize=10)
ax.legend(fontsize=8, loc="lower left")
savefig(fig, "fig_q3_4_severity.png")

In [ ]:
# ---------------- (c) class information vs reconstruction quality ----------------
rows = []
enc_cache = {}
for p in PROBES:
    key = (p["kind"], p["d"], p["seed"])
    if key not in enc_cache:
        enc_cache[key] = get_encoder(*key)
    enc = enc_cache[key]
    Xa, Xq = (Xtr01, Xte01) if p["input"] == "clean" else (XTR_NOISY, XTE_NOISY)
    Ftr, Fte = features(enc, Xa), features(enc, Xq)
    lp = p["lin"]
    lin_acc = (((Fte - lp["mu"]) / lp["sd"] @ lp["W"].T + lp["b"]).argmax(1) == yte).float().mean().item()
    knn_acc = (knn_predict(knn_normalise(Ftr, lp["mu"], lp["sd"]), ytr, knn_normalise(Fte, lp["mu"], lp["sd"]), p["k"])
               == yte).float().mean().item()
    if p["kind"] == "raw":
        mse = 0.0
    elif p["kind"] == "random":
        mse = float("nan")
    else:
        mse = float(mse_per_image(reconstruct(enc, Xte01), Xte01).mean())
    rows.append(dict(representation=p["name"], input=p["input"], d=p["d"], seed=p["seed"], test_mse_clean=mse,
                     linear_probe=lin_acc, knn_probe=knn_acc))
    del Ftr, Fte
RAW_C = pd.DataFrame(rows)

agg_rows = []
for (rep, inp), g in RAW_C.groupby(["representation", "input"], sort=False):
    m_ = g["test_mse_clean"].mean()
    agg_rows.append(dict(representation=rep, input=inp, d=int(g["d"].iloc[0]), test_mse_clean=m_,
                         test_psnr_db=psnr(m_) if np.isfinite(m_) and m_ > 0 else float("nan"),
                         linear_probe=g["linear_probe"].mean(), linear_probe_sd=sd(g["linear_probe"].values),
                         knn_probe=g["knn_probe"].mean(), knn_probe_sd=sd(g["knn_probe"].values), seeds=len(g)))
TEST_C = pd.DataFrame(agg_rows).set_index(["representation", "input"])
TEST_C.to_csv(os.path.join(OUT_DIR, "table_q3c_test_representations.csv"))
print("Table Q3-4: TEST reconstruction quality vs class information in the frozen representations")
display(TEST_C.round(4))

trained = [f"AE-{d}" for d in BOTTLENECKS] + [f"DAE-{SELECTED_D}"]
TC = TEST_C.xs("clean", level="input")
rho_lin = float(stats.spearmanr(-TC.loc[trained, "test_mse_clean"], TC.loc[trained, "linear_probe"])[0])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    rho_knn = float(stats.spearmanr(-TC.loc[trained, "test_mse_clean"], TC.loc[trained, "knn_probe"])[0])
C_DECISION = "AGREE" if rho_lin >= 0.8 else ("DISAGREE" if rho_lin <= 0 else "PARTIALLY AGREE")
print(f"Spearman(reconstruction quality, class information) over {trained}: linear probe {rho_lin:.2f}, "
      f"k-NN {rho_knn:.2f} -> reconstruction quality and class information {C_DECISION}.")
display(pd.DataFrame({"test MSE (clean)": TC.loc[trained, "test_mse_clean"],
                      "rank by reconstruction (1 = best)": TC.loc[trained, "test_mse_clean"].rank().astype(int),
                      "linear probe": TC.loc[trained, "linear_probe"],
                      "rank by linear probe (1 = best)": TC.loc[trained, "linear_probe"].rank(ascending=False).astype(int),
                      "k-NN probe": TC.loc[trained, "knn_probe"]}).round(4))

fig, ax = plt.subplots(figsize=(7.5, 4.4))
for k, nm in enumerate(trained):
    r = TC.loc[nm]
    ax.errorbar([r.test_mse_clean], [r.linear_probe], yerr=[0 if np.isnan(r.linear_probe_sd) else r.linear_probe_sd],
                marker="o", ms=8, capsize=3, color=PALETTE[k], ls="none", label=f"{nm}: linear probe")
    ax.scatter([r.test_mse_clean], [r.knn_probe], marker="D", s=40, facecolors="none", edgecolors=PALETTE[k])
    ax.annotate(nm, (r.test_mse_clean, r.linear_probe), textcoords="offset points", xytext=(6, 6), fontsize=8)
ax.set_xscale("log")
ax.axhline(TC.loc["raw pixels", "linear_probe"], color=GREY, ls="--", lw=1.2)
ax.text(0.01, TC.loc["raw pixels", "linear_probe"] + 0.005, "raw pixels (MSE = 0), linear probe",
        fontsize=7, color=GREY, transform=ax.get_yaxis_transform())
best_rand = max((f"random encoder d={d}" for d in BOTTLENECKS), key=lambda n: TC.loc[n, "linear_probe"])
ax.axhline(TC.loc[best_rand, "linear_probe"], color=GREY, ls=":", lw=1.2)
ax.text(0.01, TC.loc[best_rand, "linear_probe"] + 0.005, f"best untrained encoder ({best_rand})",
        fontsize=7, color=GREY, transform=ax.get_yaxis_transform())
ax.set_xlabel("test reconstruction MSE on clean images (log scale; lower = better reconstruction)")
ax.set_ylabel("test probe accuracy")
ax.set_title(f"Fig. Q3-5: reconstruction quality vs class information "
             f"(circles: linear probe, diamonds: k-NN; Spearman {rho_lin:.2f})", fontsize=9)
ax.legend(fontsize=7, loc="best")
savefig(fig, "fig_q3_5_recon_vs_probe.png")

## 6. Conclusions
The next cell turns the test-set results into the answers to (a), (b) and (c) using the rules fixed above. The text is saved to
`q3_outputs/conclusion.txt`.

In [ ]:
TXT = {}
TXT["a"] = (f"(a) Bottlenecks d = {BOTTLENECKS} with {', '.join(f'{count_params(ConvAE(d)):,}' for d in BOTTLENECKS)} "
            f"parameters; validation MSE {', '.join(f'{means[d]:.5f}' for d in BOTTLENECKS)}. " + SELECTION_TEXT +
            f" Test MSE of the selected AE-{SELECTED_D}: {np.mean(TEST_MSE[f'AE-{SELECTED_D}']):.5f} "
            f"(PSNR {psnr(np.mean(TEST_MSE[f'AE-{SELECTED_D}'])):.2f} dB).")
TXT["b"] = (
    f"(b) Gaussian noise sigma = {NOISE_SIGMA:g}; the DAE (same architecture as AE-{SELECTED_D}, noisy input -> clean target, "
    f"noise redrawn every batch, no labels) is compared using the fixed Q1(a) CNN. Test accuracy: clean {pct(a_clean)}, "
    f"corrupted {pct(a_noisy)}, blur {pct(a_blur)}, clean-AE reconstruction {pct(a_ae)}, DAE reconstruction {pct(a_dae)} "
    f"(ceiling: clean AE on clean images {pct(a_ceil_ae)}). The DAE recovers F = {F_rec:.2f} of the accuracy lost to noise "
    f"(95% CI of the gain {pp(CI['DAE - corrupted'][0])} to {pp(CI['DAE - corrupted'][1])}), and {F_ceil:.2f} of what "
    f"is recoverable through this reconstruction architecture. Against the clean AE it gains {pp(a_dae - a_ae)}, "
    f"and against blur {pp(a_dae - a_blur)}. Decision: denoising {B_DECISION} the information the classifier needs. " +
    ("DAE reconstructions are classified about as well as clean images." if a_clean - a_dae <= 0.01 else
     f"Of the remaining {pp(a_clean - a_dae).lstrip('+')} shortfall from clean accuracy, {pp(a_clean - a_ceil_ae).lstrip('+')} "
     "also occurs when the clean AE reconstructs clean images, so " +
     ("most of it is the classifier's penalty for smooth reconstructions, not noise left in the image."
      if (a_clean - a_ceil_ae) >= 0.5 * (a_clean - a_dae) else
      "most of it is not explained by reconstruction blur: information is genuinely lost to the noise or to the denoiser.")))
noisy_rows = TEST_C.xs("noisy", level="input")
raw_lin = float(TC.loc["raw pixels", "linear_probe"])
best_trained = max(trained, key=lambda n: TC.loc[n, "linear_probe"])
best_trained_lin = float(TC.loc[best_trained, "linear_probe"])
TXT["c"] = (
    f"(c) Linear-probe test accuracy: " +
    ", ".join(f"{nm} {pct(TC.loc[nm, 'linear_probe'])} (MSE {TC.loc[nm, 'test_mse_clean']:.5f})" for nm in trained) +
    f"; raw pixels {pct(TC.loc['raw pixels', 'linear_probe'])} at MSE 0; untrained encoders "
    + ", ".join(f"d={d}: {pct(TC.loc[f'random encoder d={d}', 'linear_probe'])}" for d in BOTTLENECKS) +
    f". Spearman between reconstruction quality and class information: {rho_lin:.2f} (linear), "
    f"{'undefined' if np.isnan(rho_knn) else f'{rho_knn:.2f}'} (k-NN). Decision: they {C_DECISION}. " +
    (f"Reference points: perfect reconstruction (raw pixels) gives {pct(raw_lin)}, "
     + ("less than" if raw_lin < best_trained_lin - 0.01 else "at least as much as")
     + f" the best trained encoder ({best_trained} {pct(best_trained_lin)}), so "
     + ("perfect reconstruction does not maximise linearly accessible class information. " if raw_lin < best_trained_lin - 0.01
        else "at the extreme, better reconstruction does go with more class information. ")
     + f"Against untrained encoders of the same width, reconstruction training changes linear-probe accuracy by "
     + ", ".join(f"d={d}: {pp(TC.loc[f'AE-{d}', 'linear_probe'] - TC.loc[f'random encoder d={d}', 'linear_probe'])}"
                 for d in BOTTLENECKS) + ". ") +
    f"On noisy inputs the DAE representation gives {pct(noisy_rows.loc[f'DAE-{SELECTED_D}', 'linear_probe'])} against "
    f"{pct(noisy_rows.loc[f'AE-{SELECTED_D}', 'linear_probe'])} for the clean AE (linear probe).")
CONCLUSION = "\n\n".join(TXT.values())
with open(os.path.join(OUT_DIR, "conclusion.txt"), "w") as f:
    f.write(CONCLUSION)
print(CONCLUSION)

### Limitations to state in the report
* **Pixel MSE favours blur.** It rewards smooth reconstructions and says nothing about what is semantically preserved, which is exactly why (c) measures class information separately.
* **The classifier is fixed and was trained on clean, sharp images.** Accuracy on reconstructions therefore mixes information loss with distribution shift. The "ceiling" rows separate the two.
* **One corruption (Gaussian noise) at one training severity.** The severity sweep is supplementary and only for test-time robustness.
* **The probes are linear models and k-NN, so they measure *accessible* class information.** A non-linear probe could find more.
* **Two seeds per autoencoder.** The probe and reconstruction differences should be read against the reported seed sd.

## 7. Reuse log

| Item | First introduced in | Used in Q3 | Notes |
|---|---|---|---|
| Stratified 45k/5k split (roll-number seed), normalisation statistics | Q1 §1 | all parts | verified identical to Q1's `split_indices.npz` when available |
| `CNN`, `count_params`, `train_model`, `predict_logits`, `df_to_lists`, `cpu_state` | Q1 §2 | classifier reload or retrain, parameter counts | code copied verbatim |
| **Fixed CNN classifier** (Q1(a) CNN, clean images, seed #0 at the LR selected in Q1) | Q1(a); also the CNN compared in Q2(a) | Q3(b) | loaded from the Q1 run cache, or retrained with identical settings if absent |
| Clean autoencoders AE-16/128/1024 | Q3(a) | Q3(b) baseline (selected AE), Q3(c) representations | — |
| DAE | Q3(b) | Q3(c) representations | — |

In [ ]:
print("Classifier source:", CLF_SOURCE)

def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o

summary = dict(roll_number=ROLL_NUMBER, seed=SEED, bottlenecks=BOTTLENECKS, selected_d=SELECTED_D,
               val_mse=means, test_mse=TEST_MSE, noise_sigma=NOISE_SIGMA, blur_sigma=BLUR_SIGMA,
               classifier=dict(lr=CLF_LR, source=CLF_SOURCE.get(CLF_LR)),
               test_classification=TEST_B["acc_mean"].to_dict(), recovered_fraction=F_rec, decision_b=B_DECISION,
               bootstrap_ci=CI, spearman_linear=rho_lin, spearman_knn=rho_knn, decision_c=C_DECISION,
               representations={f"{a}|{b}": r.to_dict() for (a, b), r in TEST_C.iterrows()}, conclusion=TXT)
with open(os.path.join(OUT_DIR, "results_summary.json"), "w") as f:
    json.dump(jsonable(summary), f, indent=2)
print("Saved to", OUT_DIR, ":", sorted(f for f in os.listdir(OUT_DIR) if f != "runs"))

In [ ]:
# ---- Record the software/hardware environment used for this run (for the report and the repository) ----
import sys, platform, subprocess, sklearn, scipy, matplotlib
ENV_USED = dict(python=sys.version.split()[0], platform=platform.platform(), torch=torch.__version__,
                torchvision=torchvision.__version__, numpy=np.__version__, pandas=pd.__version__,
                scikit_learn=sklearn.__version__, scipy=scipy.__version__, matplotlib=matplotlib.__version__,
                cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version() if torch.cuda.is_available() else None,
                device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", amp=USE_AMP)
with open(os.path.join(OUT_DIR, "environment_used.json"), "w") as f:
    json.dump(ENV_USED, f, indent=2)
with open(os.path.join(OUT_DIR, "pip_freeze.txt"), "w") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(ENV_USED)